In [11]:
import geopandas as gpd
import pandas as pd
from functools import reduce 
from functions import *

Load data

In [12]:
df_buy = pd.read_csv("datasets/omi_estimate/omi_estimate.csv")
df_buy = df_buy.drop(columns = ['mun_name','prov_name','reg_name'])

df_mun = gpd.read_file("datasets/geo_data/mun_perimeters/mun_perimeters.gpkg", layer="municipalities")
df_mun = df_mun.drop(columns = ['mun_name'])

df_zones = gpd.read_file("datasets/geo_data/omi_perimeters/omi_zone_perimeters.gpkg", layer="zones")
df_zones = df_zones.drop(columns = ['mun_name','zone_id'])
df_zones = df_zones.rename(columns = {'geometry' : 'geometry_zone'})

df_non_normalized = pd.read_csv("datasets/mun_istat_codes_non_normalized.csv")
df_non_normalized = df_non_normalized.drop(columns = ['prov_istat'])

Select 2025_S2 estimates

In [13]:
df_buy = df_buy[df_buy["year_semester"] == "2025_S2"]
add_zeroes(df_buy, 'mun_istat', 6)

,year,year_semester,semester,zone,type,condition,buy_min,buy_max,mun_istat
3642057,2025,2025_S2,2,B1,Residential housing,Normal,520,780,006003
3642058,2025,2025_S2,2,B1,Garages,Normal,600,900,006003
3642059,2025,2025_S2,2,B1,Covered parking spaces,Normal,520,780,006003
3642060,2025,2025_S2,2,B1,Uncovered parking spaces,Normal,420,630,006003
3642061,2025,2025_S2,2,B1,Warehouses,Normal,550,1100,006003
...,...,...,...,...,...,...,...,...,...
3823418,2025,2025_S2,2,B1,Warehouses,Normal,300,350,113019
3823419,2025,2025_S2,2,B1,Shops,Normal,550,750,113019
3823420,2025,2025_S2,2,B1,Offices,Normal,520,710,113019
3823421,2025,2025_S2,2,B1,Industrial buildings,Normal,360,450,113019


Merge into a single dataset

In [14]:
# Merge df_buy and df_zones on ['mun_istat','zone']
df = pd.merge(df_buy, df_zones, on = ['mun_istat', 'zone'], how = 'left')

In [15]:
add_zeroes(df_non_normalized, 'mun_istat', 6)

# Merge df, df_mun, and df_non_normalized on 'mun_istat'
dfs1 = [df, df_mun, df_non_normalized]

df = reduce(lambda left, right: pd.merge(left, right, on = ['mun_istat'], how = 'left'), dfs1)

In [16]:
df = df[[
    'mun_istat', 
    'mun_name', 
    'prov_name', 
    'reg_name',
    'zone',
    'year',
    'year_semester',
    'semester',
    'type',
    'condition',
    'buy_min',
    'buy_max',
    'geometry',
    'geometry_zone'
    ]]

df = df.rename(columns = {
    'buy_min' : 'Min. price',
    'buy_max' : 'Max. price'
})

Save data

In [18]:
gdf = gpd.GeoDataFrame(df, geometry="geometry")

In [21]:
gdf["geometry"] = gdf.geometry.simplify(tolerance=0.001)
gdf["geometry_zone"] = gdf.geometry.simplify(tolerance=0.001)
gdf.to_parquet("datasets/2025_S2_data.parquet")

GEOSException: bad allocation